# DeepDetect Experiments V2: vit_b_16
V2 pipeline: v5 skin-tone patch + phase FFT + original cleaner + dual transforms.
Spatial branch: full degradation augmentations.
Frequency branch: spatial aug only — patch selected from clean image.

Best hyperparameters from Optuna tuning (trial 13):
  backbone_lr=4.85e-05, lr=2.79e-04, freq_aux_weight=0.500,
  diversity_weight=0.058, gate_init_bias=0.215


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import torch
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell
VRAM: 25.2 GB


## Best Hyperparameters

In [2]:
BEST_BACKBONE_LR       = 4.85e-05
BEST_LR                = 2.79e-04
BEST_FREQ_AUX_WEIGHT   = 0.500
BEST_DIVERSITY_WEIGHT  = 0.058
BEST_GATE_INIT_BIAS    = 0.215


## Shared Config

In [3]:
from config import Config
from data.deepdetect_dual import get_deepdetect_dual_loaders
from experiments.train import train_v2
from experiments.evaluate import full_evaluation_v2
from experiments.baseline_spatial_only import run_spatial_only_baseline

BACKBONE = "vit_b_16"

def make_cfg(fusion_mode, frozen=False):
    cfg = Config()
    cfg.data.deepdetect_root      = "../data/raw/deep_detect/data"
    cfg.data.dataset              = "deepdetect"
    cfg.data.image_size           = 224
    cfg.data.batch_size           = 88
    cfg.data.num_workers          = 4
    cfg.backbone.name             = BACKBONE
    cfg.backbone.pretrained       = True
    cfg.backbone.img_size         = 224
    cfg.backbone.frozen           = frozen
    cfg.fusion.mode               = fusion_mode
    cfg.frequency.use_v2_pipeline = True
    cfg.frequency.cleaner_filters = 3
    cfg.frequency.patch_size      = 56
    cfg.train.early_stopping_patience = 30  
    cfg.train.epochs              = 35
    cfg.train.backbone_lr         = BEST_BACKBONE_LR
    cfg.train.lr                  = BEST_LR
    cfg.loss.freq_aux_weight      = BEST_FREQ_AUX_WEIGHT
    cfg.fusion.diversity_weight   = BEST_DIVERSITY_WEIGHT
    cfg.fusion.gate_init_bias     = BEST_GATE_INIT_BIAS
    # Spatial branch — full augmentations
    cfg.data.jpeg_aug             = True
    cfg.data.blur_aug             = True
    cfg.data.noise_aug            = True
    cfg.data.recompression_aug    = True
    cfg.data.mixed_aug            = True
    cfg.data.mixed_aug_prob       = 0.3
    cfg.experiment_name = f"dd_v2_{BACKBONE}_{fusion_mode}{'_frozen' if frozen else ''}"
    cfg.notes           = f"DeepDetect V2 · {BACKBONE} · {fusion_mode}{'_frozen' if frozen else ''}"
    return cfg

cfg_data = make_cfg("joint_only")
train_loader, val_loader, test_loader = get_deepdetect_dual_loaders(cfg_data)


Train: 76,848  Val: 13,561  Test: 21,776


## Experiment 0: Spatial-Only Baseline

In [4]:
cfg0 = make_cfg("joint_only")
cfg0.experiment_name = f"dd_v2_{BACKBONE}_spatial_only"
cfg0.notes           = f"DeepDetect V2 spatial-only baseline"
cfg0.train.epochs    = 20
spatial_acc = run_spatial_only_baseline(cfg0, train_loader, val_loader, test_loader)
print(f"\nSpatial-only floor: {spatial_acc:.1%}")


Device: cuda


Training spatial-only baseline (vit_b_16) for 20 epochs...
Train: 76,848  Val: 13,561


Epoch   1/20 | train_loss=0.4699 | val_acc=90.4%


Epoch   2/20 | train_loss=0.2091 | val_acc=94.8%


Epoch   3/20 | train_loss=0.1454 | val_acc=96.3%


...

Epoch  16/20 | train_loss=0.0048 | val_acc=98.8%


Epoch  17/20 | train_loss=0.0039 | val_acc=99.0%


Epoch  18/20 | train_loss=0.0022 | val_acc=98.9%


Epoch  19/20 | train_loss=0.0018 | val_acc=99.0%


Epoch  20/20 | train_loss=0.0012 | val_acc=99.1%

Spatial-only results (vit_b_16):
  Accuracy: 94.8%
  AUC-ROC:  0.986
  F1:       0.947
Results saved → ./results/results.csv  (dd_v2_vit_b_16_spatial_only, acc=0.9475)

Spatial-only floor: 94.8%


## Experiment 1: Joint-Only

In [5]:
cfg1 = make_cfg("joint_only")
train_v2(cfg1, train_loader, val_loader, test_loader)


Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell
Using differential lr: backbone=4.85e-05, others=2.79e-04

Experiment: dd_v2_vit_b_16_joint_only [V2 pipeline]
Backbone: vit_b_16 | Fusion: joint_only | Frozen: False
Epochs: 35 | LR: 0.000279 | Batch: 88
Spatial branch: full aug | Frequency branch: spatial aug only



Epoch   1/35 | train_loss=0.5872 | val_acc=98.1% | val_auc=0.999 | val_f1=0.980 | best=0.0% | patience=0/30
  -> Saved checkpoint (epoch 1, val_acc=98.1%)


Epoch   2/35 | train_loss=0.4265 | val_acc=98.2% | val_auc=0.999 | val_f1=0.980 | best=98.1% | patience=0/30
  -> Saved checkpoint (epoch 2, val_acc=98.2%)


Epoch   3/35 | train_loss=0.3983 | val_acc=98.1% | val_auc=0.999 | val_f1=0.979 | best=98.2% | patience=0/30
  -> Saved checkpoint (epoch 3, val_acc=98.1%)


Epoch   4/35 | train_loss=0.3762 | val_acc=98.1% | val_auc=0.999 | val_f1=0.979 | best=98.2% | patience=0/30
  -> Saved checkpoint (epoch 4, val_acc=98.1%)


...

Epoch  32/35 | train_loss=0.2600 | val_acc=99.7% | val_auc=1.000 | val_f1=0.996 | best=99.6% | patience=0/30
  -> Saved checkpoint (epoch 32, val_acc=99.7%)


Epoch  33/35 | train_loss=0.2595 | val_acc=99.6% | val_auc=1.000 | val_f1=0.996 | best=99.7% | patience=0/30
  -> Saved checkpoint (epoch 33, val_acc=99.6%)


Epoch  34/35 | train_loss=0.2587 | val_acc=99.6% | val_auc=1.000 | val_f1=0.996 | best=99.7% | patience=0/30
  -> Saved checkpoint (epoch 34, val_acc=99.6%)


Epoch  35/35 | train_loss=0.2602 | val_acc=99.7% | val_auc=1.000 | val_f1=0.996 | best=99.7% | patience=0/30
  -> Saved checkpoint (epoch 35, val_acc=99.7%)

Training complete. Best val accuracy: 99.7%
Results will be logged to CSV after full_evaluation_v2().


0.9966816606444953

In [6]:
results1 = full_evaluation_v2(
    cfg1,
    checkpoint_path=f"./checkpoints/best_{cfg1.experiment_name}.pt",
    test_loader=test_loader,
    dataset_type="deepdetect",
)

Loaded checkpoint: ./checkpoints/best_dd_v2_vit_b_16_joint_only.pt

EVALUATION — dd_v2_vit_b_16_joint_only
Backbone: vit_b_16 | Fusion: joint_only | Frozen: False
  Joint accuracy:     91.1%
  AUC-ROC:            0.979
  F1:                 0.915
  Spatial-only:       91.2%
  Freq-only:          71.2%
  Delta (Δ):          -0.1%  (freq branch contribution)
Results saved → ./results/results.csv  (dd_v2_vit_b_16_joint_only, acc=0.9112)



## Experiment 2: Scalar Fusion

In [7]:
cfg2 = make_cfg("scalar")
train_v2(cfg2, train_loader, val_loader, test_loader)

Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell
Using differential lr: backbone=4.85e-05, others=2.79e-04

Experiment: dd_v2_vit_b_16_scalar [V2 pipeline]
Backbone: vit_b_16 | Fusion: scalar | Frozen: False
Epochs: 35 | LR: 0.000279 | Batch: 88
Spatial branch: full aug | Frequency branch: spatial aug only



Epoch   1/35 | train_loss=0.5832 | val_acc=96.9% | val_auc=0.999 | val_f1=0.965 | best=0.0% | patience=0/30
  -> Saved checkpoint (epoch 1, val_acc=96.9%)


Epoch   2/35 | train_loss=0.4313 | val_acc=98.7% | val_auc=0.999 | val_f1=0.986 | best=96.9% | patience=0/30
  -> Saved checkpoint (epoch 2, val_acc=98.7%)


Epoch   3/35 | train_loss=0.3941 | val_acc=99.0% | val_auc=0.999 | val_f1=0.989 | best=98.7% | patience=0/30
  -> Saved checkpoint (epoch 3, val_acc=99.0%)


Epoch   4/35 | train_loss=0.3754 | val_acc=98.0% | val_auc=0.999 | val_f1=0.978 | best=99.0% | patience=0/30
  -> Saved checkpoint (epoch 4, val_acc=98.0%)


...

Epoch  32/35 | train_loss=0.2558 | val_acc=99.6% | val_auc=1.000 | val_f1=0.995 | best=99.6% | patience=0/30
  -> Saved checkpoint (epoch 32, val_acc=99.6%)


Epoch  33/35 | train_loss=0.2559 | val_acc=99.6% | val_auc=1.000 | val_f1=0.996 | best=99.6% | patience=0/30
  -> Saved checkpoint (epoch 33, val_acc=99.6%)


Epoch  34/35 | train_loss=0.2554 | val_acc=99.6% | val_auc=1.000 | val_f1=0.995 | best=99.6% | patience=0/30
  -> Saved checkpoint (epoch 34, val_acc=99.6%)


Epoch  35/35 | train_loss=0.2551 | val_acc=99.6% | val_auc=1.000 | val_f1=0.995 | best=99.6% | patience=0/30
  -> Saved checkpoint (epoch 35, val_acc=99.6%)

Training complete. Best val accuracy: 99.6%
Results will be logged to CSV after full_evaluation_v2().


0.9962392153970946

In [8]:
results2 = full_evaluation_v2(
    cfg2,
    checkpoint_path=f"./checkpoints/best_{cfg2.experiment_name}.pt",
    test_loader=test_loader,
    dataset_type="deepdetect",
)

Loaded checkpoint: ./checkpoints/best_dd_v2_vit_b_16_scalar.pt

EVALUATION — dd_v2_vit_b_16_scalar
Backbone: vit_b_16 | Fusion: scalar | Frozen: False
  Joint accuracy:     93.9%
  AUC-ROC:            0.981
  F1:                 0.939
  Spatial-only:       93.9%
  Freq-only:          72.2%
  Delta (Δ):          -0.1%  (freq branch contribution)
Results saved → ./results/results.csv  (dd_v2_vit_b_16_scalar, acc=0.9388)



## Experiment 3: Gating Fusion

In [9]:
cfg3 = make_cfg("gating")
train_v2(cfg3, train_loader, val_loader, test_loader)


Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell
Using differential lr: backbone=4.85e-05, others=2.79e-04

Experiment: dd_v2_vit_b_16_gating [V2 pipeline]
Backbone: vit_b_16 | Fusion: gating | Frozen: False
Epochs: 35 | LR: 0.000279 | Batch: 88
Spatial branch: full aug | Frequency branch: spatial aug only



Epoch   1/35 | train_loss=0.6331 | val_acc=96.5% | val_auc=0.999 | val_f1=0.961 | best=0.0% | patience=0/30
  -> Saved checkpoint (epoch 1, val_acc=96.5%)


Epoch   2/35 | train_loss=0.4797 | val_acc=98.8% | val_auc=0.999 | val_f1=0.987 | best=96.5% | patience=0/30
  -> Saved checkpoint (epoch 2, val_acc=98.8%)


Epoch   3/35 | train_loss=0.4436 | val_acc=98.6% | val_auc=0.999 | val_f1=0.985 | best=98.8% | patience=0/30
  -> Saved checkpoint (epoch 3, val_acc=98.6%)


Epoch   4/35 | train_loss=0.4360 | val_acc=97.8% | val_auc=0.999 | val_f1=0.976 | best=98.8% | patience=0/30
  -> Saved checkpoint (epoch 4, val_acc=97.8%)


...

Epoch  32/35 | train_loss=0.3073 | val_acc=99.7% | val_auc=1.000 | val_f1=0.997 | best=99.7% | patience=0/30
  -> Saved checkpoint (epoch 32, val_acc=99.7%)


Epoch  33/35 | train_loss=0.3025 | val_acc=99.7% | val_auc=1.000 | val_f1=0.997 | best=99.7% | patience=0/30
  -> Saved checkpoint (epoch 33, val_acc=99.7%)


Epoch  34/35 | train_loss=0.3071 | val_acc=99.7% | val_auc=1.000 | val_f1=0.997 | best=99.7% | patience=0/30
  -> Saved checkpoint (epoch 34, val_acc=99.7%)


Epoch  35/35 | train_loss=0.3060 | val_acc=99.7% | val_auc=1.000 | val_f1=0.996 | best=99.7% | patience=0/30
  -> Saved checkpoint (epoch 35, val_acc=99.7%)

Training complete. Best val accuracy: 99.7%
Results will be logged to CSV after full_evaluation_v2().


0.997419069390163

In [10]:
results3 = full_evaluation_v2(
    cfg3,
    checkpoint_path=f"./checkpoints/best_{cfg3.experiment_name}.pt",
    test_loader=test_loader,
    dataset_type="deepdetect",
)


Loaded checkpoint: ./checkpoints/best_dd_v2_vit_b_16_gating.pt

EVALUATION — dd_v2_vit_b_16_gating
Backbone: vit_b_16 | Fusion: gating | Frozen: False
  Joint accuracy:     95.5%
  AUC-ROC:            0.989
  F1:                 0.954
  Spatial-only:       95.5%
  Freq-only:          71.2%
  Delta (Δ):          -0.1%  (freq branch contribution)

  Gate distribution:
    entropy:  0.959 nats  (OK)
    mean:     0.583
    variance: 0.0031
Results saved → ./results/results.csv  (dd_v2_vit_b_16_gating, acc=0.9546)



## Experiment 4: Gating Frozen

In [11]:
cfg4 = make_cfg("gating", frozen=True)
train_v2(cfg4, train_loader, val_loader, test_loader)


Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell
Using differential lr: backbone=4.85e-05, others=2.79e-04

Experiment: dd_v2_vit_b_16_gating_frozen [V2 pipeline]
Backbone: vit_b_16 | Fusion: gating | Frozen: True
Epochs: 35 | LR: 0.000279 | Batch: 88
Spatial branch: full aug | Frequency branch: spatial aug only



Epoch   1/35 | train_loss=0.9025 | val_acc=84.2% | val_auc=0.921 | val_f1=0.826 | best=0.0% | patience=0/30
  -> Saved checkpoint (epoch 1, val_acc=84.2%)


Epoch   2/35 | train_loss=0.7937 | val_acc=83.0% | val_auc=0.944 | val_f1=0.782 | best=84.2% | patience=0/30
  -> Saved checkpoint (epoch 2, val_acc=83.0%)


Epoch   3/35 | train_loss=0.7392 | val_acc=88.0% | val_auc=0.952 | val_f1=0.866 | best=84.2% | patience=0/30
  -> Saved checkpoint (epoch 3, val_acc=88.0%)


Epoch   4/35 | train_loss=0.7090 | val_acc=87.2% | val_auc=0.946 | val_f1=0.863 | best=88.0% | patience=0/30
  -> Saved checkpoint (epoch 4, val_acc=87.2%)


...

Epoch  32/35 | train_loss=0.4704 | val_acc=91.4% | val_auc=0.979 | val_f1=0.901 | best=92.3% | patience=0/30
  -> Saved checkpoint (epoch 32, val_acc=91.4%)


Epoch  33/35 | train_loss=0.4658 | val_acc=91.6% | val_auc=0.980 | val_f1=0.904 | best=92.3% | patience=0/30
  -> Saved checkpoint (epoch 33, val_acc=91.6%)


Epoch  34/35 | train_loss=0.4649 | val_acc=91.9% | val_auc=0.980 | val_f1=0.908 | best=92.3% | patience=0/30
  -> Saved checkpoint (epoch 34, val_acc=91.9%)


Epoch  35/35 | train_loss=0.4617 | val_acc=91.7% | val_auc=0.980 | val_f1=0.906 | best=92.3% | patience=0/30
  -> Saved checkpoint (epoch 35, val_acc=91.7%)

Training complete. Best val accuracy: 92.3%
Results will be logged to CSV after full_evaluation_v2().


0.9227195634540226

In [12]:
results4 = full_evaluation_v2(
    cfg4,
    checkpoint_path=f"./checkpoints/best_{cfg4.experiment_name}.pt",
    test_loader=test_loader,
    dataset_type="deepdetect",
)


Loaded checkpoint: ./checkpoints/best_dd_v2_vit_b_16_gating_frozen.pt

EVALUATION — dd_v2_vit_b_16_gating_frozen
Backbone: vit_b_16 | Fusion: gating | Frozen: True
  Joint accuracy:     89.5%
  AUC-ROC:            0.962
  F1:                 0.887
  Spatial-only:       82.3%
  Freq-only:          69.9%
  Delta (Δ):          +7.2%  (freq branch contribution)

  Gate distribution:
    entropy:  2.889 nats  (OK)
    mean:     0.595
    variance: 0.0705

  No warning signs triggered.
Results saved → ./results/results.csv  (dd_v2_vit_b_16_gating_frozen, acc=0.8955)



## Summary

In [13]:
import pandas as pd
df = pd.read_csv("./results/results.csv", on_bad_lines='skip')
mask = df["experiment_name"].str.startswith(f"dd_v2_{BACKBONE}")
cols = ["experiment_name", "fusion", "frozen", "accuracy", "auc_roc", "f1", "gate_entropy"]
print(df[mask][cols].to_string(index=False))


             experiment_name     fusion  frozen  accuracy  auc_roc     f1  gate_entropy
 dd_v2_vit_b_16_spatial_only joint_only   False    0.9475   0.9863 0.9471        0.0000
   dd_v2_vit_b_16_joint_only joint_only   False    0.9112   0.9791 0.9145        0.0000
       dd_v2_vit_b_16_scalar     scalar   False    0.9388   0.9812 0.9394        0.0000
       dd_v2_vit_b_16_gating     gating   False    0.9546   0.9891 0.9544        0.9594
dd_v2_vit_b_16_gating_frozen     gating    True    0.8955   0.9623 0.8869        2.8890
